In [1]:
import os

# Fix: must be set before importing TensorFlow to avoid protobuf incompatibility crash
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
os.environ.pop("PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION_VERSION", None)
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import random
import numpy as np
import pandas as pd

import tensorflow as tf
from math import ceil

from tensorflow.keras.layers import Dense, Input, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from tensorflow.keras.optimizers import Adam

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism(True)
except Exception:
    pass

try:
    tf.config.threading.set_intra_op_parallelism_threads(0)  # let TF decide
    tf.config.threading.set_inter_op_parallelism_threads(0)  # let TF decide
except Exception:
    pass

tf.keras.backend.clear_session()
print("TF version:", tf.__version__)



2026-05-11 23:38:05.183030: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778542685.202308      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778542685.207705      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

TF version: 2.18.0


In [2]:
gm_exp = tf.Variable(3.0, dtype=tf.float32)


def generalized_mean_pool_2d(X):
    pool = (
        tf.reduce_mean(tf.abs(X ** (gm_exp)), axis=[1, 2], keepdims=False) + 1.0e-7
    ) ** (1.0 / gm_exp)
    return pool


def create_model(input_shape):
    inp = Input(shape=input_shape)

    x_model = ResNet50(
        weights="imagenet",
        include_top=False,
        input_tensor=inp,
        pooling=None,
        classes=None,
    )
    for layer in x_model.layers:
        layer.trainable = True

    lambda_layer = Lambda(generalized_mean_pool_2d)
    lambda_layer.trainable_weights.extend([gm_exp])

    x = lambda_layer(x_model.output)
    out = Dense(5, activation="softmax", name="plan_diseases")(x)
    model = Model(inputs=x_model.input, outputs=out)
    return model




2026-05-11 23:38:17.297263: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [3]:
path = "/kaggle/input/cassava-leaf-disease-classification/"
train_csv_path = os.path.join(path, "train.csv")
sample_sub_path = os.path.join(path, "sample_submission.csv")
train_img_dir = os.path.join(path, "train_images") + "/"
test_img_dir = os.path.join(path, "test_images") + "/"
train_tfrec_dir = os.path.join(path, "train_tfrecords")
test_tfrec_dir = os.path.join(path, "test_tfrecords")

train_df = pd.read_csv(train_csv_path)
sample_submission = pd.read_csv(sample_sub_path)

print("train_df:", train_df.shape, "sample_submission:", sample_submission.shape)
print(train_df.head())



train_df: (18721, 2) sample_submission: (2676, 2)
         image_id  label
0  1000015157.jpg      0
1  1000201771.jpg      3
2   100042118.jpg      1
3  1000723321.jpg      1
4  1000812911.jpg      3


In [4]:
IMG_H = 224
IMG_W = 224

# Fix: test TFRecords may not contain "target"; make test schema independent from train schema.
_TRAIN_FEATURES = {
    "image": tf.io.FixedLenFeature([], tf.string),
    "image_name": tf.io.FixedLenFeature([], tf.string),
    "target": tf.io.FixedLenFeature([], tf.int64),
}
_TEST_FEATURES = {
    "image": tf.io.FixedLenFeature([], tf.string),
    "image_name": tf.io.FixedLenFeature([], tf.string),
}


@tf.function(reduce_retracing=True)
def _decode_resize_preprocess_from_bytes(img_bytes):
    img = tf.io.decode_jpeg(img_bytes, channels=3, dct_method="INTEGER_FAST")
    img = tf.image.resize(
        img, (IMG_H, IMG_W), method=tf.image.ResizeMethod.BILINEAR, antialias=False
    )
    img = tf.cast(img, tf.float32)
    img = preprocess_input(img)
    return img


@tf.function(reduce_retracing=True)
def _parse_train_example(example_proto):
    ex = tf.io.parse_single_example(example_proto, _TRAIN_FEATURES)
    img = _decode_resize_preprocess_from_bytes(ex["image"])
    y = tf.cast(ex["target"], tf.int32)
    return img, y


@tf.function(reduce_retracing=True)
def _parse_test_example_with_name(example_proto):
    ex = tf.io.parse_single_example(example_proto, _TEST_FEATURES)
    img = _decode_resize_preprocess_from_bytes(ex["image"])
    name = ex["image_name"]
    return img, name


@tf.function(reduce_retracing=True)
def _parse_image_label_example(example_proto):
    ex = tf.io.parse_single_example(example_proto, _TRAIN_FEATURES)
    img = _decode_resize_preprocess_from_bytes(ex["image"])
    y = tf.cast(ex["target"], tf.int32)
    return img, y


def _list_tfrecords(dir_path, prefix):
    if not tf.io.gfile.exists(dir_path):
        return []
    files = tf.io.gfile.glob(os.path.join(dir_path, f"{prefix}*.tfrec"))
    return sorted(files)


def _build_dataset_from_tfrecords(
    tfrecord_files,
    batch_size=16,
    training=False,
    seed=42,
    include_names=False,
):
    ds = tf.data.TFRecordDataset(
        tfrecord_files,
        num_parallel_reads=tf.data.AUTOTUNE,
        buffer_size=8 * 1024 * 1024,
    )

    options = tf.data.Options()
    options.experimental_deterministic = True
    options.threading.private_threadpool_size = 0
    options.threading.max_intra_op_parallelism = 0
    options.experimental_slack = True
    ds = ds.with_options(options)

    if training:
        ds = ds.map(
            _parse_train_example,
            num_parallel_calls=tf.data.AUTOTUNE,
            deterministic=True,
        )
        ds = ds.shuffle(2048, seed=seed, reshuffle_each_iteration=True)
        ds = ds.repeat()
    else:
        ds = ds.map(
            _parse_test_example_with_name,
            num_parallel_calls=tf.data.AUTOTUNE,
            deterministic=True,
        )
        if not include_names:
            ds = ds.map(
                lambda img, name: img,
                num_parallel_calls=tf.data.AUTOTUNE,
                deterministic=True,
            )

    ds = ds.apply(tf.data.experimental.ignore_errors())
    ds = ds.batch(batch_size, drop_remainder=False)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds


def _build_image_label_dataset_from_tfrecords(
    tfrecord_files,
    batch_size=16,
):
    ds = tf.data.TFRecordDataset(
        tfrecord_files,
        num_parallel_reads=tf.data.AUTOTUNE,
        buffer_size=8 * 1024 * 1024,
    )

    options = tf.data.Options()
    options.experimental_deterministic = True
    options.threading.private_threadpool_size = 0
    options.threading.max_intra_op_parallelism = 0
    options.experimental_slack = True
    ds = ds.with_options(options)

    ds = ds.map(
        _parse_image_label_example,
        num_parallel_calls=tf.data.AUTOTUNE,
        deterministic=True,
    )
    ds = ds.apply(tf.data.experimental.ignore_errors())
    ds = ds.batch(batch_size, drop_remainder=False)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds


@tf.function(reduce_retracing=True)
def _decode_resize_preprocess(path):
    img_bytes = tf.io.read_file(path)
    return _decode_resize_preprocess_from_bytes(img_bytes)


def _build_dataset_from_paths(
    image_paths,
    labels=None,
    batch_size=16,
    shuffle=False,
    seed=42,
):
    image_paths = tf.convert_to_tensor(image_paths, dtype=tf.string)

    if labels is not None:
        labels = tf.convert_to_tensor(labels, dtype=tf.int32)
        ds = tf.data.Dataset.from_tensor_slices((image_paths, labels))
    else:
        ds = tf.data.Dataset.from_tensor_slices(image_paths)

    options = tf.data.Options()
    options.experimental_deterministic = True
    options.threading.private_threadpool_size = 0
    options.threading.max_intra_op_parallelism = 0
    options.experimental_slack = True
    ds = ds.with_options(options)

    if shuffle:
        buffer_size = int(image_paths.shape[0])
        ds = ds.shuffle(
            buffer_size=buffer_size,
            seed=seed,
            reshuffle_each_iteration=True,
        )

    if labels is not None:

        def _map_fn(path, y):
            return _decode_resize_preprocess(path), y

        ds = ds.map(_map_fn, num_parallel_calls=tf.data.AUTOTUNE, deterministic=True)
    else:

        def _map_img(path):
            return _decode_resize_preprocess(path)

        ds = ds.map(_map_img, num_parallel_calls=tf.data.AUTOTUNE, deterministic=True)

    ds = ds.apply(tf.data.experimental.ignore_errors())
    ds = ds.batch(batch_size, drop_remainder=False)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds




In [5]:
path_weights = "../input/weights/"
has_external_weights = os.path.isdir(path_weights) and len(os.listdir(path_weights)) > 0
print("External weights found:", has_external_weights)

test_tfrecord_files = _list_tfrecords(test_tfrec_dir, "ld_test")
train_tfrecord_files = _list_tfrecords(train_tfrec_dir, "ld_train")
print(
    "Found test tfrecords:",
    len(test_tfrecord_files),
    "train tfrecords:",
    len(train_tfrecord_files),
)



External weights found: False
Found test tfrecords: 2 train tfrecords: 14


In [6]:
input_shape = (224, 224, 3)
batch_size = 16

X_test = sample_submission.copy()
if "label" in X_test.columns:
    X_test = X_test.drop(columns=["label"])

use_tfrecord_test = len(test_tfrecord_files) > 0
if use_tfrecord_test:
    ds_test = _build_dataset_from_tfrecords(
        test_tfrecord_files,
        batch_size=batch_size,
        training=False,
        seed=SEED,
        include_names=True,
    )
else:
    test_image_paths = (test_img_dir + X_test["image_id"].astype(str)).to_numpy()
    ds_test = _build_dataset_from_paths(
        test_image_paths,
        labels=None,
        batch_size=batch_size,
        shuffle=False,
        seed=SEED,
    )



Instructions for updating:
Use `tf.data.Dataset.ignore_errors` instead.


In [7]:
all_pred = []

model = create_model(input_shape)

model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
    jit_compile=True,
    run_eagerly=False,
    steps_per_execution=16,
)

if has_external_weights:
    models = sorted(os.listdir(path_weights))
    weight_paths = [os.path.join(path_weights, w) for w in models]
else:
    weight_paths = [None]

for wpath in weight_paths:
    if wpath is not None:
        model.load_weights(wpath)

    if use_tfrecord_test:
        preds_batches = []
        names_batches = []
        batch_count = 0
        for img_batch, name_batch in ds_test:
            batch_count += 1
            preds_batches.append(model.predict_on_batch(img_batch))
            names_batches.append(name_batch.numpy())

        # Safety: if TFRecord parsing failed and produced zero batches, fallback to image paths.
        if batch_count == 0:
            print(
                "WARNING: TFRecord test dataset yielded 0 batches; falling back to test_images/ loading."
            )
            use_tfrecord_test = False
            test_image_paths = (
                test_img_dir + X_test["image_id"].astype(str)
            ).to_numpy()
            ds_test_fallback = _build_dataset_from_paths(
                test_image_paths,
                labels=None,
                batch_size=batch_size,
                shuffle=False,
                seed=SEED,
            )
            preds = model.predict(ds_test_fallback, verbose=1)
            preds = preds[: sample_submission.shape[0]]
        else:
            preds = np.concatenate(preds_batches, axis=0)
            names = np.concatenate(names_batches, axis=0)

            if names.dtype.kind in ("S", "O"):
                names = np.array(
                    [
                        (
                            n.decode("utf-8")
                            if isinstance(n, (bytes, bytearray))
                            else str(n)
                        )
                        for n in names
                    ],
                    dtype=str,
                )
            else:
                names = names.astype(str)

            preds = preds[: sample_submission.shape[0]]
            names = names[: sample_submission.shape[0]]

            pred_df = pd.DataFrame({"image_id": names})
            pred_df["row_id"] = np.arange(pred_df.shape[0], dtype=np.int32)

            merged = sample_submission[["image_id"]].merge(
                pred_df, on="image_id", how="left", sort=False, validate="one_to_one"
            )
            row_idx = merged["row_id"].to_numpy(
                np.float64
            )  # allows NaN detection robustly

            if np.any(pd.isna(row_idx)):
                missing = int(np.sum(pd.isna(row_idx)))
                raise RuntimeError(
                    f"Failed to align TFRecord names with sample_submission for {missing} rows."
                )

            row_idx = row_idx.astype(np.int64)
            preds = preds[row_idx]
    else:
        preds = model.predict(ds_test, verbose=1)
        preds = preds[: sample_submission.shape[0]]

    all_pred.append(preds)

if len(all_pred) == 0:
    raise RuntimeError(
        "No predictions were generated (all_pred is empty); cannot build submission."
    )



94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


/usr/local/lib/python3.11/dist-packages/keras/src/models/functional.py:237: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['keras_tensor']
Received: inputs=Tensor(shape=(16, 224, 224, 3))
  warnings.warn(msg)
I0000 00:00:1778542706.337920      60 service.cc:148] XLA service 0x7fff1400e7b0 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778542706.337956      60 service.cc:156]   StreamExecutor device (0): Host, Default Version
I0000 00:00:1778542707.466932      60 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
/usr/local/lib/python3.11/dist-packages/keras/src/models/functional.py:237: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['keras_tensor']
Received: inputs=Tensor(shape=(None, 224, 224, 3))
  warnings.warn(msg)


In [8]:
sum_pred = np.mean(np.stack(all_pred, axis=0), axis=0)

diagnos = np.argmax(sum_pred, axis=1).astype(int).tolist()

sample_submission["label"] = diagnos
print(sample_submission.head())
print("Label value counts:\n", sample_submission["label"].value_counts())



         image_id  label
0  1234294272.jpg      0
1  1234332763.jpg      0
2  1234375577.jpg      0
3  1234555380.jpg      2
4  1234571117.jpg      2
Label value counts:
 label
0    1915
2     562
4     173
1      26
Name: count, dtype: int64


In [9]:
sample_submission[["image_id", "label"]].to_csv("submission.csv", index=False)
print(
    "Wrote submission.csv with shape:", sample_submission[["image_id", "label"]].shape
)
print("submission.csv preview:")
print(pd.read_csv("submission.csv").head())

Wrote submission.csv with shape: (2676, 2)
submission.csv preview:
         image_id  label
0  1234294272.jpg      0
1  1234332763.jpg      0
2  1234375577.jpg      0
3  1234555380.jpg      2
4  1234571117.jpg      2
